# Traditional ML Baseline (Member 2)
**Research Objective:** Establish the baseline accuracy and latency ceiling for the triage classification task using classical ML and TF-IDF.
**Targets:** `category`, `priority`, `sentiment`
**Approach:** 
- Dataset: Loads the pre-split, locked `train_split.csv` and `test_split.csv` ensuring fair comparison against other models.
- Feature Engineering: Combine text features, clean text, TF-IDF vectorization.
- Hyperparameter Tuning: Optuna optimization for LinearSVC (or Random Forest).
- Evaluation: Custom classification report matching the exact requested format.

In [ ]:
!pip install optuna pandas scikit-learn numpy

import pandas as pd
import numpy as np
import optuna
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Load the Pre-Split Datasets
# Ensure you have uploaded BOTH 'train_split.csv' and 'test_split.csv' to Colab
df_train_raw = pd.read_csv('train_split.csv')
df_test_raw = pd.read_csv('test_split.csv')

print(f"Loaded train_split.csv: {len(df_train_raw)} tickets")
print(f"Loaded test_split.csv: {len(df_test_raw)} tickets")

In [ ]:
# 2. Feature Engineering Pipeline (Applied independently to prevent data leakage)

def preprocess_data(df):
    df = df.copy()
    
    # Drop unneeded columns as requested
    cols_to_drop = ['channel', 'platform', 'source', 'ticket_id']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    
    # Combine issue_description and product to enrich the TF-IDF context
    df['text_feature'] = df['product'].astype(str) + " - " + df['issue_description'].astype(str)
    
    # Clean text (lowercasing)
    df['text_feature'] = df['text_feature'].str.lower()
    return df

df_train = preprocess_data(df_train_raw)
df_test = preprocess_data(df_test_raw)

# Define targets
targets = ['category', 'priority', 'sentiment']

X_train = df_train['text_feature']
y_train = df_train[targets]

X_test = df_test['text_feature']
y_test = df_test[targets]


In [ ]:
# TF-IDF Vectorization
# NOTE: We only fit on X_train to prevent data leakage from the test set!
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Optuna Hyperparameter Tuning for each Target
# We use LinearSVC as it handles high-dimensional sparse TF-IDF data very fast and effectively.

best_models = {}

def optimize_target(target_name):
    print(f"--- Tuning for {target_name.upper()} ---")
    
    def objective(trial):
        # Hyperparameters to tune
        c_param = trial.suggest_float('C', 0.01, 10.0, log=True)
        
        clf = LinearSVC(C=c_param, random_state=42, class_weight='balanced')
        clf.fit(X_train_vec, y_train[target_name])
        
        preds = clf.predict(X_test_vec)
        # Optimize for Macro F1
        f1 = f1_score(y_test[target_name], preds, average='macro')
        return f1
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10) # Set to higher (e.g., 50) for better optimization
    
    print(f"Best trial for {target_name}: {study.best_value}")
    print(f"Best params for {target_name}: {study.best_params}\n")
    
    # Train best model
    best_clf = LinearSVC(C=study.best_params['C'], random_state=42, class_weight='balanced')
    best_clf.fit(X_train_vec, y_train[target_name])
    best_models[target_name] = best_clf

for target in targets:
    optimize_target(target)

In [ ]:
# Custom Evaluation Output exactly matching the requested format

def print_custom_report(y_true, y_pred, target_name):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    macro_prec = precision_score(y_true, y_pred, average='macro')
    macro_rec = recall_score(y_true, y_pred, average='macro')
    
    print("================================================================================")
    print(f"*** {target_name.upper()}")
    print("================================================================================")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Macro Precision: {macro_prec:.4f}")
    print(f"Macro Recall: {macro_rec:.4f}")
    print("")
    
    # Classification report
    rep = classification_report(y_true, y_pred, digits=2)
    print(rep)
    print("\n")

for target in targets:
    preds = best_models[target].predict(X_test_vec)
    print_custom_report(y_test[target], preds, target)

## ⚙️ Hyperparameter Optimization Checklist (How to tweak this further)
If you want to push the baseline accuracy even higher in Colab, change these values:

1. **`n_trials=10`** -> Increase this to `50` or `100` in the Optuna block. This gives the optimizer more time to find the absolute perfect parameters.
2. **`max_features=5000`** -> Change this to `10000` or `None` in the `TfidfVectorizer` block to include more words (will use more RAM).
3. **`ngram_range=(1, 2)`** -> Change this to `(1, 3)` to allow the TF-IDF vectorizer to capture 3-word phrases (e.g., "not very happy" instead of just "not very").
4. **Change the Model** -> Currently it uses `LinearSVC`. You can import `from sklearn.ensemble import RandomForestClassifier` and test it in the objective function to see if non-linear tree splits beat the SVM hyperplane.